# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset, using the `mlcroissant` library and referencing all entities by their `@id` fields.

### Dataset Source
The dataset is described using a Croissant schema at the following URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access dataset metadata (direct attribute access, not as a dict)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Dataset version: {dataset.metadata.version}\n")
# Optionally display published date, license and cite-as
print(f"Published on: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(dataset.metadata, 'license', 'N/A')}")
print(f"Cite as: {getattr(dataset.metadata, 'citeAs', 'N/A')}")


## 2. Data Overview
Review available record sets, their IDs, and the available fields for each record set. All references use each entity's `@id`.

In [ ]:
from pprint import pprint

record_sets = dataset.record_sets
print(f"\nRecord sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    # Show name/description if available
    name = rs.get('name', '')
    description = rs.get('description', '')
    if name: print(f"  Name: {name}")
    if description: print(f"  Description: {description}")
    # List fields/columns
    if 'field' in rs:
        print(f"  Fields (@id):")
        if isinstance(rs['field'], list):
            for field in rs['field']:
                if isinstance(field, dict):
                    print(f"    - {field.get('@id', str(field))}")
                else:
                    print(f"    - {field}")
        else:
            print(f"    - {rs['field']}")
    elif 'column' in rs:
        print(f"  Columns (@id):")
        for col in rs['column']:
            print(f"    - {col.get('@id', str(col))}")
    print()

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s from above.

In [ ]:
# List all record set @id values for convenience
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"RecordSet @ids: {record_set_ids}\n")

# Load records from each record set into a pandas DataFrame
dfs = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dfs[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from RecordSet {record_set_id}")
        print(f"Columns: {dfs[record_set_id].columns.tolist()}")
    else:
        print(f"No records found for RecordSet {record_set_id}")

# For interactive exploration, pick the main data table (by row count or name)
main_record_set = None
max_rows = 0
for rsid, df in dfs.items():
    if len(df) > max_rows:
        main_record_set = rsid
        max_rows = len(df)
if main_record_set:
    print(f"\nUsing main_record_set = {main_record_set}")
    print(dfs[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Process and analyze the data from the primary record set, using only column `@id` to reference fields as specified by the schema.

In [ ]:
# Replace this example with actual @id field names from the main record set.
# We'll try to infer a numeric field, and a group by field.
df = dfs[main_record_set]

# List columns
print(f"Columns (@id): {df.columns.tolist()}")

# Attempt to find a numeric field:
numeric_candidate = None
for c in df.columns:
    try:
        s = pd.to_numeric(df[c], errors='coerce')
        if s.notna().sum() > 0 and s.dtype in [float, int]:
            numeric_candidate = c
            break
    except Exception:
        continue

if numeric_candidate is None:
    raise ValueError("No numeric field found in the record set for analysis.")
print(f"Using numeric field for analysis: {numeric_candidate}")

# Filter records above mean value for demonstration
values = pd.to_numeric(df[numeric_candidate], errors='coerce')
threshold = values.mean()
filtered_df = df[values > threshold].copy()
print(f"Filtered records in '{numeric_candidate}' above mean ({threshold:.2f}): {len(filtered_df)} of {len(df)}")

# Normalize numeric field
filtered_df[f"{numeric_candidate}_normalized"] = (values[values > threshold] - values[values > threshold].mean()) / values[values > threshold].std()
print(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

# Choose a group field (categorical, if available)
group_field = None
for c in df.columns:
    if c != numeric_candidate and df[c].dtype == object:
        n_unique = df[c].nunique()
        if 1 < n_unique < df.shape[0] // 2:
            group_field = c
            break
if group_field:
    print(f"Using group field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_candidate].mean().reset_index()
    print(grouped_df.head())
else:
    print("No suitable group field for aggregation.")

## 5. Visualization
Visualize the distribution of the main numeric field and, if available, summary by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of numeric field
plt.figure(figsize=(7, 4))
sns.histplot(pd.to_numeric(df[numeric_candidate], errors='coerce').dropna(), bins=30)
plt.title(f"Distribution of {numeric_candidate}")
plt.xlabel(numeric_candidate)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# If group_field was found, show average per group
if group_field:
    plt.figure(figsize=(8, 4))
    sns.barplot(x=group_field, y=numeric_candidate, data=grouped_df)
    plt.title(f"Average {numeric_candidate} by {group_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we:
- Accessed the FAIR^2 dataset using its Croissant schema URL and `mlcroissant`
- Inspected metadata and discovered available record sets and fields by `@id`
- Loaded record set(s) into pandas DataFrames for analysis
- Performed basic filtering and normalization on a selected numeric field
- (If available) Grouped and visualized data by a key categorical field

This demonstrates FAIR record set and field discovery, canonical data loading, and reproducible EDA using only Croissant `@id` references. For further analysis, refine field and record set selection and reference the Croissant schema for metadata about specific columns or record set details.
